# Prueba de Modelos XLM-RoBERTa desde Hugging Face

Carga modelos desde Hugging Face y realiza pruebas con dataset TASS.

In [9]:
import pandas as pd
from transformers import pipeline

# Mapeo de etiquetas (del dataset y del modelo)
label_map = {
    'N': 'Negativo', 'NEU': 'Neutral', 'P': 'Positivo',
    'LABEL_0': 'Negativo', 'LABEL_1': 'Neutral', 'LABEL_2': 'Positivo'
}

# Cargar dataset
df_sample = pd.read_csv('input/TASS/cr1-tass.csv').sample(n=100, random_state=42)
print(f"✓ Cargadas {len(df_sample)} muestras aleatorias")

✓ Cargadas 100 muestras aleatorias


In [10]:
# Cargar modelos desde Hugging Face
pipe_bs8 = pipeline("text-classification", model="Yenreh/xlm-roberta-large-tass-sentiment-bs8")
pipe_bs16 = pipeline("text-classification", model="Yenreh/xlm-roberta-large-tass-sentiment-bs16")
print("✓ Modelos cargados")

Device set to use cuda:0
Device set to use cuda:0


✓ Modelos cargados


## Pruebas con ejemplos personalizados

In [11]:
ejemplos = [
    "La selección de fútbol de Colombia perdió todos los partidos y lo sacaron del mundial",
    "La clase de PLN sigue siendo uno de los cursos de MAIN",
    "El curso de PLN es excelente",
    "Esto es terrible, no me gusta nada",
    "Hoy es un día normal, sin nada especial"
]

print("EJEMPLOS PERSONALIZADOS")
print("=" * 80)
for ejemplo in ejemplos:
    pred_bs8 = pipe_bs8(ejemplo)[0]
    pred_bs16 = pipe_bs16(ejemplo)[0]
    print(f"\n {ejemplo}")
    print(f"   BS8:  {label_map[pred_bs8['label']]} ({pred_bs8['score']:.3f}) | BS16: {label_map[pred_bs16['label']]} ({pred_bs16['score']:.3f})")

EJEMPLOS PERSONALIZADOS

 La selección de fútbol de Colombia perdió todos los partidos y lo sacaron del mundial
   BS8:  Negativo (0.992) | BS16: Negativo (0.381)

 La clase de PLN sigue siendo uno de los cursos de MAIN
   BS8:  Positivo (0.968) | BS16: Negativo (0.381)

 El curso de PLN es excelente
   BS8:  Positivo (0.979) | BS16: Positivo (0.741)

 Esto es terrible, no me gusta nada
   BS8:  Negativo (0.994) | BS16: Negativo (0.381)

 Hoy es un día normal, sin nada especial
   BS8:  Neutral (0.648) | BS16: Negativo (0.381)


## Pruebas con ejemplos del dataset TASS

In [12]:
# Tomar ejemplos del dataset por cada clase
print("\nEJEMPLOS DEL DATASET TASS (3 por clase)")
print("=" * 80)

for etiqueta in ['N', 'NEU', 'P']:
    ejemplos_clase = df_sample[df_sample['label'] == etiqueta].head(3)
    for _, row in ejemplos_clase.iterrows():
        pred_bs8 = pipe_bs8(row['sentencia original'])[0]
        pred_bs16 = pipe_bs16(row['sentencia original'])[0]
        print(f"\n {row['sentencia original'][:70]}...")
        print(f"   Real: {label_map[row['label']]} | BS8: {label_map[pred_bs8['label']]} ({pred_bs8['score']:.3f}) | BS16: {label_map[pred_bs16['label']]} ({pred_bs16['score']:.3f})")


EJEMPLOS DEL DATASET TASS (3 por clase)

 Encima estoy en mal de amores tio....
   Real: Negativo | BS8: Negativo (0.994) | BS16: Negativo (0.381)

 @sebatramp Acá también, Seba ???? Para peor el sismógrafo no da los da...
   Real: Negativo | BS8: Negativo (0.994) | BS16: Negativo (0.382)

 @Brenluarte nunca te calles, si le molesta a alguien es su problema, n...
   Real: Negativo | BS8: Negativo (0.993) | BS16: Negativo (0.381)

 @aniu96 @Chuz_CM Yo no puedo ingerir bebidas alcoholicas, ni marihuana...
   Real: Neutral | BS8: Neutral (0.987) | BS16: Negativo (0.382)

 Mientras mi jefe no se entere todo ok lol Los demás si tenemos sentido...
   Real: Neutral | BS8: Neutral (0.991) | BS16: Negativo (0.382)

 @KatyKlav se siente, ya está dicho...
   Real: Neutral | BS8: Neutral (0.985) | BS16: Negativo (0.381)

 @_rvng En Castalla (Alicante) muy oportunamente, a primeros de mes cel...
   Real: Positivo | BS8: Positivo (0.974) | BS16: Negativo (0.380)

 @Tokpelotas @Carlbozal +1 y coinci